# Notebook 2 -- Relations entre capteurs et premiere tentative de reconstruction
### Corrélation, régression multivariée et projection PCA comme étalon statique

> ****Note de provenance.**** Les notebooks de recherche originaux seront présents dans le répertoire du projet, accompagnés des données et des fichiers nécessaires aux expériences. Le présent notebook fait uniquement office de **notebook d'analyse et de synthèse** : il ne remplace pas les notebooks originaux et ne constitue pas une nouvelle implémentation des expérimentations.
>
> Le code, les résultats et les sorties présentés dans ce notebook sont analysés à partir des travaux réalisés dans les notebooks originaux. Aucun résultat n'a été recalculé ni inventé : lorsqu'une information n'était pas déterminable à partir des notebooks ou des données disponibles, cela est signalé explicitement plutôt que comblé. Ce notebook n'est donc pas conçu pour être exécuté isolément de haut en bas ; il a pour objectif de **documenter, structurer, comparer et interpréter les résultats déjà obtenus**, ainsi que de mettre en évidence la progression scientifique entre les différentes approches étudiées.


## Objectif

Avant d'introduire un modele dynamique (SINDy, notebook 4) ou un modele appris de representation
(Dictionary Learning, notebook 3 ; XGBoost, notebooks 6-7), ce notebook cherche a repondre a une
question plus simple : **une relation purement statique entre les capteurs disponibles a
l'instant t et le capteur cible au meme instant permet-elle deja de reconstruire le signal
manquant ?** Il introduit egalement la notion de *matrice de transfert* H, evoquee comme premiere
piste conceptuelle dans le cahier des charges de ce memoire, et precise ce qui, dans ce notebook,
s'en rapproche et ce qui n'en est pas.

## Motivation

Le notebook 1 a etabli qu'une redondance statistique existe entre plusieurs capteurs et la cible
`ACR_Mot.PV` (correlations de l'ordre de 0,29 a 0,85 selon le capteur). La question qui se pose
naturellement est de savoir jusqu'ou cette seule redondance statique permet d'aller, avant
d'investir dans des methodes plus couteuses supposant une dynamique (SINDy) ou une representation
apprise (Dictionary Learning, XGBoost). Une regression lineaire multivariee et une projection sur
composantes principales (PCA) constituent la reponse la plus directe et la plus interpretable a
cette question, et serviront d'etalon (*baseline*) pour tous les notebooks suivants.

## Hypothese

Une relation lineaire instantanee entre les capteurs disponibles et le capteur cible,
`y_cible(t) ~ f(y_autres(t))`, capture une partie utile de l'information necessaire a la
reconstruction. Cette hypothese ne suppose aucune dynamique temporelle ni aucune non-linearite :
elle isole volontairement le cas le plus simple, afin que toute methode plus complexe introduite
ensuite doive demontrer un gain reel par rapport a elle.

## Precision methodologique : matrice de transfert H, regression et PCA ne sont pas la meme chose

Le cahier des charges de ce memoire insiste, a juste titre, sur le fait qu'une matrice de
reconstruction statistique ne doit jamais etre presentee comme equivalente a une veritable matrice
de transfert physique H. Il est utile de clarifier ce point des ce notebook, puisque c'est ici
qu'apparaissent les premieres matrices de ce type :

- Une **matrice de transfert physique H**, au sens strict, relierait les capteurs a partir d'un
  modele mecanique de propagation des vibrations (masse, raideur, amortissement, distance,
  frequence, attenuation du materiau). Aucun notebook du corpus ne construit une telle matrice a
  partir d'un modele physique explicite.
- La **matrice de coefficients de la regression lineaire** ci-dessous est une matrice de
  **reconstruction statistique** : elle relie les capteurs disponibles a la cible en minimisant une
  erreur quadratique sur les donnees observees, sans aucune contrainte physique. Deux capteurs
  fortement correles mais sans lien mecanique direct produiraient exactement le meme type de
  coefficient qu'une veritable relation de transfert.
- La **matrice de chargement de la PCA** est une matrice de **projection statistique** : elle
  capture les directions de variance maximale communes a l'ensemble des capteurs (cible incluse
  en Train), pas une relation causale ou dirigee entre eux.

Cette distinction sera reprise et approfondie au notebook 3 (Dictionary Learning), ou la meme
question se pose avec encore plus d'acuite pour la matrice de dictionnaire D.

## Methodologie

Les capteurs disponibles sont classes par correlation absolue avec la cible, calculee **sur Train
uniquement** (apres le decoupage temporel du notebook 1, pour eviter toute fuite d'information
du Validation/Test vers la selection de variables) :

In [20]:
corr_with_target = df_train[AVAILABLE_SENSORS].corr()[TARGET_SENSOR].drop(TARGET_SENSOR).sort_values(key=abs, ascending=False)
print("Corrélation (Train) des capteurs disponibles avec la cible, triée par |corr| :")
print(corr_with_target)

SENSOR_CORR_MIN_THRESHOLD = 0.0   # mettre par ex. 0.05 pour exclure les capteurs quasi-non-corrélés
selected_sensors = [c for c in OTHER_SENSORS if abs(corr_with_target.get(c, 1.0)) >= SENSOR_CORR_MIN_THRESHOLD]
print(f"\nCapteurs retenus pour SSI ({len(selected_sensors)}) :", selected_sensors)

ALL_SSI_SENSORS = [TARGET_SENSOR] + [c for c in selected_sensors]  # cible incluse en Train pour identifier C_missing


Corrélation (Train) des capteurs disponibles avec la cible, triée par |corr| :
ACR_Pmp.SV     0.846127
Temp.PV        0.824737
ACR_Pmp.TV     0.802134
ACR_Mot.TV     0.777371
ACR_Pmp.PV     0.669205
Barometer      0.441663
ACR_Mot.SV     0.366265
Temperature    0.359642
Pres.PV        0.293063
Name: ACR_Mot.PV, dtype: float64

Capteurs retenus pour SSI (9) : ['ACR_Mot.SV', 'ACR_Mot.TV', 'ACR_Pmp.PV', 'ACR_Pmp.SV', 'ACR_Pmp.TV', 'Pres.PV', 'Temp.PV', 'Barometer', 'Temperature']


Les 9 capteurs disponibles sont tous conserves comme predicteurs (aucun seuil d'exclusion
applique, `SENSOR_CORR_MIN_THRESHOLD = 0`), avec des correlations allant de 0,29 (`Pres.PV`) a
0,85 (`ACR_Pmp.SV`) -- une redondance moderee a forte pour la moitie superieure du classement,
plus faible pour la moitie inferieure.

Trois methodes statiques sont definies (la quatrieme baseline codee dans la meme cellule source,
*SSI + Kalman sans SINDy*, ne prend tout son sens qu'a partir du notebook 5, une fois SSI
introduit ; elle est neanmoins montree ici telle qu'elle apparait dans le code original, pour ne
pas casser la cellule source) :

- **Interpolation lineaire** temporelle du capteur cible (ignore totalement les autres capteurs --
  sert de plancher de comparaison).
- **Regression lineaire multivariee**, apprise sur Train, reliant les capteurs disponibles a la
  cible.
- **PCA**, ajustee sur Train (cible incluse), puis reconstruction de la cible par resolution aux
  moindres carres des scores compatibles avec les seuls capteurs disponibles.

In [46]:
# --- Baseline 1 : interpolation temporelle ---
def baseline_interpolation(y_true, mask):
    y_masked = y_true.copy()
    y_masked[mask] = np.nan
    y_hat = pd.Series(y_masked).interpolate(method="linear", limit_direction="both").values
    return y_hat

# --- Baseline 2 : régression multivariée (Train) ---
from sklearn.linear_model import LinearRegression
reg_model = LinearRegression()
reg_model.fit(train_s[selected_sensors].values, train_s[TARGET_SENSOR].values)

def baseline_regression(Y_others):
    return reg_model.predict(Y_others)

# --- Baseline 3 : reconstruction PCA (Train, incluant la cible) ---
from sklearn.decomposition import PCA
PCA_N_COMPONENTS = min(len(ALL_SSI_SENSORS), max(2, SSI_ORDER))
pca_model = PCA(n_components=PCA_N_COMPONENTS)
pca_model.fit(train_s[ALL_SSI_SENSORS].values)   # colonnes = [TARGET_SENSOR] + selected_sensors

def baseline_pca(Y_others):
    '''Reconstruit la cible en résolvant, dans le sous-espace PCA, les scores compatibles
    avec les capteurs disponibles uniquement, puis en reconstruisant la composante cible.'''
    components = pca_model.components_               # (k, m_all)
    mean_ = pca_model.mean_
    idx_target = 0                                     # TARGET_SENSOR est en 1ère colonne de ALL_SSI_SENSORS
    idx_others = list(range(1, len(ALL_SSI_SENSORS)))
    W_others = components[:, idx_others]               # (k, m_other)
    mean_others = mean_[idx_others]
    # scores = argmin || W_others^T s - (y_others - mean_others) ||  (moindres carrés)
    scores, *_ = np.linalg.lstsq(W_others.T, (Y_others - mean_others).T, rcond=None)
    y_hat = (components[:, idx_target] @ scores) + mean_[idx_target]
    return y_hat

# --- Baseline 4 : SSI + Kalman SANS SINDy (dynamique linéaire discrète brute) ---
def f_linear(x):
    '''Approxime la dynamique continue équivalente à x_{k+1}=Ax_k : dx/dt = (A - I)/dt * x'''
    return (A - np.eye(SSI_ORDER)) @ x / DT_APPROX

def baseline_ssi_kalman_no_sindy(Y_others, x0=None):
    if x0 is None:
        x0 = ssi_estimate_states(Y_others[:SSI_BLOCK_ROWS], C_obs_full, A, min(SSI_BLOCK_ROWS, len(Y_others)))[0] \
             if len(Y_others) >= SSI_BLOCK_ROWS else np.zeros(SSI_ORDER)
    Q = EKF_Q_SCALE * np.eye(SSI_ORDER)
    R = EKF_R_SCALE * np.eye(Y_others.shape[1])
    ekf_lin = ExtendedKalmanFilter(f_linear, C_obs_full, Q, R, DT_APPROX)
    x = x0.copy(); P = 0.1*np.eye(SSI_ORDER)
    X_hat = np.zeros((len(Y_others), SSI_ORDER))
    for k in range(len(Y_others)):
        x, P = ekf_lin.step(x, P, Y_others[k])
        X_hat[k] = x
    return (X_hat @ C_target_row.T).ravel()

print("Baselines définies : interpolation, régression, PCA, SSI+Kalman (sans SINDy).")


Baselines définies : interpolation, régression, PCA, SSI+Kalman (sans SINDy).


## Resultats

Les trois methodes ci-dessus sont evaluees dans le notebook 5, sur la meme fenetre continue
manquante que la methode dynamique SSI + SINDy + Kalman, ce qui permet une comparaison directe a
perimetre egal. Par souci de lisibilite, les chiffres obtenus pour la fenetre principale (20 % du
Test, capteur cible totalement masque sur cette fenetre) sont deja rapportes ici -- ils sont repris
integralement, sans modification, au notebook 5 :

| Methode | RMSE | R² | Correlation | Peak RMSE |
|---|---|---|---|---|
| Interpolation lineaire | 0,837 | -0,22 | 0,686 | 0,000 |
| **Regression lineaire multivariee** | **0,662** | **0,234** | **0,915** | 0,713 |


*(Unites : signal standardise, ecart-type 1 sur Train. Source : notebook `NB-proto`, tableau de
comparaison finale, fenetre continue mediane du Test.)*

## Interpretation

La regression lineaire multivariee est, de facon inattendue, **la methode la plus performante de
tout le corpus sur cette fenetre precise** parmi toutes celles testees dans les notebooks 2, 5 et
6 (elle sera battue uniquement par certaines variantes XGBoost du notebook 6, evaluees cependant
sur un jeu de donnees et un protocole differents -- voir la mise en garde du notebook 1). C'est un
resultat important a ne pas minimiser : une relation statique instantanee, sans aucune dynamique
ni aucune notion d'etat latent, capture ici une part non negligeable de l'information utile
(R² = 0,234, correlation = 0,915).


Le R² de 0,234 de la regression, meme s'il est le meilleur du trio, reste loin de 1 : une relation
statique instantanee laisse donc une part substantielle de la variance du signal inexpliquee. Cela
motive directement la question du notebook suivant.

## Limites

Aucune des deux methodes de ce notebook n'utilise d'information temporelle (ni retard, ni
etat latent, ni dynamique) : chacune traite chaque instant independamment des autres. Aucune
n'utilise non plus d'information physique sur la propagation des vibrations. Le bon score de la
regression pourrait donc refleter une correlation instantanee forte specifique a ce jeu de
donnees et a cette fenetre d'evaluation, sans garantie de generalisation a un autre regime de
fonctionnement de la machine ou a un evenement transitoire plus marque -- cette question est
justement reprise plus loin (peak RMSE de la regression = 0,713, deja nettement moins bon que son
RMSE global, un premier signe que la reconstruction des pics est plus difficile que celle du
regime courant).

## Conclusion

## Transition vers l'approche suivante

Ce que ce notebook a permis de comprendre : une relation purement statique et instantanee
(regression lineaire) capture deja une part utile de l'information (R² = 0,234). 
Ce qu'il n'a pas permis de resoudre : R² = 0,234 reste loin de
1, et l'erreur sur les pics (0,713) est deja visiblement plus mauvaise que l'erreur globale
(0,662) -- signe qu'un modele purement statique et instantane risque de mal se comporter
precisement la ou cela compte le plus, sur les evenements transitoires. Une question se pose
alors : peut-on apprendre une representation plus riche des signaux, capable de mieux capturer
les motifs recurrents multi-capteurs, sans pour autant supposer une dynamique explicite ? C'est la
piste exploree au notebook 3 avec le Dictionary Learning.